<a href="https://colab.research.google.com/github/rainforest01-coder/ESAA_files/blob/OB/0306_%EC%84%B8%EC%85%98_%EB%AA%A8%EB%8D%B8%ED%9B%88%EB%A0%A8_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


수백만 개의 특성이 있는 경우에는 확률적 경사 하강법(SGD) 또는 미니배치 경사 하강법 같은 경사하강법 기반 선형 회귀 알고리즘을 사용하는 것이 적합합니다.

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

검증 오차가 계속 증가 → 과대적합 발생

해결 방법 → 조기 종료(Early Stopping), 규제 추가, 모델 단순화, 데이터 증가.

### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

문제: 높은 편향 (underfitting)

이유: 훈련 오차와 검증 오차가 둘 다 높고 비슷함

해결: α를 줄여 규제를 약하게 만들어 모델을 더 유연하게 한다.

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

선형 회귀 → 릿지:	과대적합 방지  
릿지 → 라쏘:	자동 특성 선택  
라쏘 → 엘라스틱넷:	상관된 특성이 많을 때 안정적인 선택  

### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [2]:
import numpy as np
from sklearn.datasets import load_iris

# 1. 데이터 로드
iris = load_iris()
X = iris["data"]
y = iris["target"]

# bias 추가
X = np.c_[np.ones((X.shape[0], 1)), X]

# 2. 원-핫 인코딩
n_classes = len(np.unique(y))
Y = np.eye(n_classes)[y]

# 3. 데이터 분리
np.random.seed(42)
shuffle_idx = np.random.permutation(len(X))

X = X[shuffle_idx]
Y = Y[shuffle_idx]

train_size = int(len(X) * 0.8)

X_train = X[:train_size]
Y_train = Y[:train_size]

X_val = X[train_size:]
Y_val = Y[train_size:]

# 4. 소프트맥스 함수
def softmax(logits):
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)

# 5. 손실 함수
def cross_entropy(y_true, y_pred):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-10), axis=1))

# 6. 초기화
n_features = X_train.shape[1]
Theta = np.random.randn(n_features, n_classes)

eta = 0.1
n_epochs = 1000

best_val_loss = np.inf
patience = 10
wait = 0

# 7. 배치 경사 하강법
for epoch in range(n_epochs):

    logits = X_train @ Theta
    y_proba = softmax(logits)

    gradients = (1/len(X_train)) * X_train.T @ (y_proba - Y_train)
    Theta -= eta * gradients

    # 검증 손실
    val_logits = X_val @ Theta
    val_proba = softmax(val_logits)
    val_loss = cross_entropy(Y_val, val_proba)

    print(epoch, val_loss)

    # 조기 종료
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_theta = Theta.copy()
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping")
            break

Theta = best_theta

0 3.2667546784464934
1 1.6504721316761661
2 1.0102977913211968
3 1.0484057442239814
4 0.9566182216350001
5 1.0404526433878554
6 0.9458741393082928
7 1.024622224365342
8 0.9343959065500551
9 1.0003615589081838
10 0.9202193831349307
11 0.9736673206195903
12 0.9055517230169561
13 0.9483799477256447
14 0.8915305730961202
15 0.9254218541943016
16 0.8783746867730792
17 0.9046911041742927
18 0.8660163912165617
19 0.8858878180407718
20 0.8543436185275909
21 0.8687207486326708
22 0.8432547092974507
23 0.8529417909984996
24 0.8326647348160114
25 0.8383451071555382
26 0.8225033715896234
27 0.8247606597967712
28 0.8127121805174194
29 0.812047710165417
30 0.8032423255556759
31 0.800089248446887
32 0.7940527471674014
33 0.7887874458654089
34 0.7851087114634384
35 0.7780600136562378
36 0.7763806603821213
37 0.767837310029918
38 0.7678432993998784
39 0.7580600438668372
40 0.7594748709584607
41 0.7486774474048892
42 0.7512565726518974
43 0.7396458155616384
44 0.743172088380225
45 0.7309273318426577
46 